In [6]:
import torch
print("Pytorch version:", torch.__version__)

Pytorch version: 2.12.0
time: 556 μs (started: 2026-05-18 00:01:55 +06:00)


In [7]:

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

mps
time: 64.4 ms (started: 2026-05-18 00:01:57 +06:00)


In [12]:
# Create tensors on GPU
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)

# Operations run on GPU automatically
z = torch.matmul(x, y)
print(f"✓ GPU computation done: {z.shape}")

✓ GPU computation done: torch.Size([1000, 1000])
time: 170 ms (started: 2026-05-18 00:03:50 +06:00)


In [4]:
load_ext autotime

time: 66.1 μs (started: 2026-05-18 00:01:49 +06:00)


In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pathlib
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns
import os
import random
import shutil
import PIL as pil
import cv2
import matplotlib as mpl
import tqdm
from torchinfo import summary
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

In [10]:
batch_size = 32
img_height = 224
img_width = 224

class_names = ["Chickenpox", "Cowpox", "Healthy", "HFMD", "Measles", "Monkeypox"]

time: 569 μs (started: 2026-05-18 00:02:29 +06:00)


In [2]:
from pathlib import Path
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# Now this will work
dataset_path = Path.home() / 'Jupyter' / 'MIRA_TEST_1' / 'Dataset' / 'Original Images'

print(f"Dataset path: {dataset_path}")
print(f"Exists: {dataset_path.exists()}")

# Rest of your code...

Dataset path: /Users/abdullahalhossain/Jupyter/MIRA_TEST_1/Dataset/Original Images
Exists: True


In [11]:
transform = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                        [0.229, 0.224, 0.225])  # ImageNet normalization
])

time: 2.68 ms (started: 2026-05-18 00:02:32 +06:00)


In [15]:
train_path = dataset_path / 'FOLDS' / 'fold1' / 'Train'
valid_path = dataset_path / 'FOLDS' / 'fold1' / 'Valid'
test_path = dataset_path / 'FOLDS' / 'fold1' / 'Test'
print(f"Train path exists: {train_path.exists()}")
print(f"Valid path exists: {valid_path.exists()}")
print(f"Test path exists: {test_path.exists()}")

Train path exists: True
Valid path exists: True
Test path exists: True
time: 2.02 ms (started: 2026-05-18 00:06:52 +06:00)


In [16]:


train_ds = datasets.ImageFolder(root=train_path, transform=transform)
valid_ds = datasets.ImageFolder(root=valid_path, transform=transform)
test_ds = datasets.ImageFolder(root=test_path, transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, num_workers=12,
                                          shuffle=True)

valid_loader = torch.utils.data.DataLoader(valid_ds, batch_size=batch_size, num_workers=12,
                                          shuffle=False)

test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, num_workers=12,
                                         shuffle=False)

time: 294 ms (started: 2026-05-18 00:07:44 +06:00)


/Users/abdullahalhossain/Jupyter/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [17]:
def model_summary(model:torch.nn.Module,
                  input_size:tuple):
  return summary(model=model,
                input_size=input_size,
                col_names=["input_size", "output_size", "num_params", "trainable"],
                col_width=20,
                row_settings=['var_names'])

time: 1.01 ms (started: 2026-05-18 00:08:03 +06:00)


In [19]:
num_classes = 5

model = torchvision.models.resnet50(weights='DEFAULT') # Here you can change the backbone model name and write that bacbone model name which you want to use.

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential() # Here you need to write your chosen backbone model last layer name (Eg. For Densenet121 the last layer name is "classifier")

num_features = 2048 # Here you need to write your chosen backbone model last number of features (Eg. For Densenet121 the last number of features is "1024")

custom_classifier = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.ReLU(),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, num_classes)
)

model.fc = custom_classifier # Here you need to write your chosen backbone model last layer name (Eg. For Densenet121 the last layer name is "classifier")

model_summary(model=model,
              input_size=(batch_size,3,224,224))



Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
ResNet (ResNet)                          [32, 3, 224, 224]    [32, 5]              --                   Partial
├─Conv2d (conv1)                         [32, 3, 224, 224]    [32, 64, 112, 112]   (9,408)              False
├─BatchNorm2d (bn1)                      [32, 64, 112, 112]   [32, 64, 112, 112]   (128)                False
├─ReLU (relu)                            [32, 64, 112, 112]   [32, 64, 112, 112]   --                   --
├─MaxPool2d (maxpool)                    [32, 64, 112, 112]   [32, 64, 56, 56]     --                   --
├─Sequential (layer1)                    [32, 64, 56, 56]     [32, 256, 56, 56]    --                   False
│    └─Bottleneck (0)                    [32, 64, 56, 56]     [32, 256, 56, 56]    --                   False
│    │    └─Conv2d (conv1)               [32, 64, 56, 56]     [32, 64, 56, 56]     (4,096)              False
│    │    

time: 4.38 s (started: 2026-05-18 00:40:48 +06:00)


In [20]:
def evaluate(model, data_loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    loss = running_loss / len(data_loader)
    accuracy = correct / total

    return loss, accuracy

time: 1.12 ms (started: 2026-05-18 00:41:26 +06:00)


In [21]:
def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs):
    best_val_loss = float('inf')
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_accuracy = correct / total

        val_loss, val_accuracy = evaluate(model, val_loader, criterion)

        print(f"Epoch {epoch + 1}/{epochs}, "
              f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            torch.save(model.state_dict(), filepath)
            print(f"Model weights saved to {filepath}")
            best_val_loss = val_loss

time: 2.17 ms (started: 2026-05-18 00:42:12 +06:00)


In [22]:
train(model, train_loader=train_loader, val_loader=valid_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, epochs=epochs)

NameError: name 'criterion' is not defined

time: 51.1 ms (started: 2026-05-18 00:42:36 +06:00)
